# 🔬 08 · 方差分析（ANOVA）与实验设计

> 比较多组均值时，两两 t 检验会累积假阳性。**ANOVA（方差分析）**
> 一次性检验"所有组均值是否相同"，是实验数据的核心分析工具。
> 本节：单因素 ANOVA → 双因素（含交互）→ 效应量 → 多重比较校正。

## 📋 本节要解决的问题

- ANOVA 的思想：为什么叫"方差"分析？组间方差 vs 组内方差
- F 统计量怎么算？手写单因素 ANOVA
- 双因素 ANOVA 的交互作用怎么理解？
- 效应量（η²）怎么算？ANOVA 通过后还要做什么？

## 📖 论文配图：单因素三组的雨云图：组间变异 vs 组内变异

> **图源**：Ott VL et al. (2025) *Informative data visualization with raincloud plots in JASP*, Behavior Research Methods（开放获取，CC-BY 4.0）图 3。

![单因素三组的雨云图：组间变异 vs 组内变异](../figures/anova_raincloud.jpg)

> **怎么读这张图**：雨云图（raincloud）= 左侧散点/雨滴 + 密度 + 箱线，展示三组（G1/G2/G3）数据的分布。ANOVA 的本质：**组间均值差异（信号）相对组内波动（噪声）是否足够大**。这张图让人直观看到三组的重叠程度，对应本节 F = 组间均方/组内均方的直觉。


## 1️⃣ ANOVA 的思想（为什么用方差比较均值）

把总变异拆成两部分：
- **组间变异 SSB**：各组均值与总均值之差（反映组间真实差异）
- **组内变异 SSW**：各组内部观测与组均值之差（反映随机噪声/个体差异）

$$F = \frac{SSB / (k-1)}{SSW / (n-k)} = \frac{MSB}{MSW}$$
H₀ 为真时 F ≈ 1；组间差异大时 F >> 1。

> **面试**：为什么不以"均值差"直接做检验？→ 需除以噪声尺度（组内方差），
> F 统计量是"信噪比"。

In [1]:
import numpy as np
from scipy import stats as st

def anova_oneway(groups):
    k = len(groups)
    all_data = np.concatenate(groups)
    grand_mean = all_data.mean()
    n = len(all_data)
    # 组间平方和
    ssb = sum(len(g) * (np.mean(g) - grand_mean) ** 2 for g in groups)
    # 组内平方和
    ssw = sum(((g - np.mean(g)) ** 2).sum() for g in groups)
    dfb, dfw = k - 1, n - k
    msb, msw = ssb / dfb, ssw / dfw
    f = msb / msw
    p = 1 - st.f.cdf(f, dfb, dfw)
    return f, dfb, dfw, p, ssb, ssw

rng = np.random.RandomState(12)
# 三组：A 均值 10，B 均值 12，C 均值 14（组内 σ=2）
groups = [rng.normal(10, 2, 30), rng.normal(12, 2, 30), rng.normal(14, 2, 30)]
f, dfb, dfw, p, ssb, ssw = anova_oneway(groups)
print('单因素 ANOVA: F=%.3f  p=%.6f' % (f, p))
# 对照 scipy
f_s, p_s = st.f_oneway(*groups)
print('scipy 对照:   F=%.3f  p=%.6f' % (f_s, p_s))
assert abs(f - f_s) < 1e-9
assert p < 0.05
print('✅ 手写 ANOVA 与 scipy 一致，三组均值显著不同')

单因素 ANOVA: F=29.428  p=0.000000
scipy 对照:   F=29.428  p=0.000000
✅ 手写 ANOVA 与 scipy 一致，三组均值显著不同


## 2️⃣ 效应量 η²（不要只看 p 值）

$$\eta^2 = \frac{SSB}{SSB + SSW}$$
- η²=0.01 小效应、0.06 中效应、0.14 大效应（经验基准）

> **为什么报告效应量**：p 值受样本量影响，η² 是"解释了多少变异"的标准化度量。

In [2]:
def eta_squared(ssb, ssw):
    return ssb / (ssb + ssw)

# 用上节数据
eta2 = eta_squared(ssb, ssw)
print('η²=%.3f → %s' % (eta2, '大效应' if eta2 > 0.14 else ('中效应' if eta2 > 0.06 else '小效应')))
print('→ 组间差异解释了 %.1f%% 的总体变异' % (eta2 * 100))
assert 0.1 < eta2 < 0.5
print('✅ 效应量计算完成')

η²=0.404 → 大效应
→ 组间差异解释了 40.4% 的总体变异
✅ 效应量计算完成


## 3️⃣ 双因素 ANOVA 与交互作用

两个因素 A、B 及其**交互** A×B：
- 主效应：A 单独的影响、B 单独的影响
- 交互效应：A 的效果是否依赖 B 的水平

> **经典例子**：药物效果 × 性别。药对男有效、对女无效 → 交互显著。
> 有显著交互时必须分开看简单效应。

In [3]:
from scipy import stats as st

# 用 statsmodels 做双因素 ANOVA（含交互）
import statsmodels.api as sm
from statsmodels.formula.api import ols

rng = np.random.RandomState(25)
n_per = 20
# 因素A: 方法(旧/新), 因素B: 场景(线上/线下), 新方法只在线上有效 → 交互
data = []
for a in ['旧', '新']:
    for b in ['线上', '线下']:
        base = 50
        if a == '新' and b == '线上':
            base += 15
        if a == '新':
            base += 2
        vals = rng.normal(base, 4, n_per)
        for v in vals:
            data.append({'method': a, 'scene': b, 'score': v})

df_data = __import__('pandas', fromlist=['DataFrame']).DataFrame(data)
model = ols('score ~ C(method) * C(scene)', data=df_data).fit()
an = sm.stats.anova_lm(model, typ=2)
print(an.round(3))
p_inter = an.loc['C(method):C(scene)', 'PR(>F)']
print('交互项 p=%.4f %s' % (p_inter, '显著 → 新方法效果依赖场景 ✅' if p_inter < 0.05 else '不显著'))
print('✅ 双因素 ANOVA（含交互）完成')

                      sum_sq    df       F  PR(>F)
C(method)           2025.068   1.0  93.818     0.0
C(scene)            1185.054   1.0  54.902     0.0
C(method):C(scene)  1342.446   1.0  62.193     0.0
Residual            1640.463  76.0     NaN     NaN
交互项 p=0.0000 显著 → 新方法效果依赖场景 ✅
✅ 双因素 ANOVA（含交互）完成


## 4️⃣ ANOVA 后：多重比较校正

ANOVA 显著 ≠ 哪两组不同。后续两两比较需校正：
- **Tukey HSD**：专门为 ANOVA 后所有两两比较设计，控制 FWER
- Bonferroni：α/m（最保守）

> **面试**：为什么 ANOVA 后不能直接做 m 次 t 检验？
> → 累积假阳性：独立 t 检验 m 次，至少一个假阳性 = 1-(1-α)^m

In [4]:
# Tukey HSD 演示（用 statsmodels）
from statsmodels.stats.multicomp import pairwise_tukeyhsd

rng = np.random.RandomState(33)
groups_data = []
labels = []
for mu, lab in [(10, 'A'), (12, 'B'), (14, 'C')]:
    vals = rng.normal(mu, 2, 30)
    groups_data.extend(vals)
    labels.extend([lab] * 30)
res = pairwise_tukeyhsd(np.array(groups_data), np.array(labels), alpha=0.05)
print(res.summary())
print('→ A-B、A-C、B-C 差异均显著（reject=True）✅')

Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj  lower  upper  reject
--------------------------------------------------
     A      B   2.1838 0.0003 0.9144 3.4533   True
     A      C   4.6304    0.0  3.361 5.8999   True
     B      C   2.4466    0.0 1.1772 3.7161   True
--------------------------------------------------
→ A-B、A-C、B-C 差异均显著（reject=True）✅


## 5️⃣ 实验设计与 ANOVA 的前提

ANOVA 三大前提：
1. **独立性**：各组观测相互独立（实验设计保证）
2. **正态性**：各组内近似正态（大样本可放宽）
3. **方差齐性**：各组方差相近（Levene 检验）

> **面试**：方差齐性不满足怎么办？→ Welch ANOVA（不假定等方差），
> 或 Kruskal-Wallis（非参数）。

In [5]:
from scipy import stats as st
# Levene 方差齐性检验 + Kruskal 非参数对比
rng = np.random.RandomState(44)
g1 = rng.normal(10, 1, 30); g2 = rng.normal(12, 3, 30); g3 = rng.normal(14, 5, 30)
W, p_levene = st.levene(g1, g2, g3)
print('Levene 方差齐性: W=%.3f p=%.3f → %s' % (W, p_levene, '方差不等（需 Welch/非参数）' if p_levene < 0.05 else '方差齐'))
H, p_kw = st.kruskal(g1, g2, g3)
print('Kruskal-Wallis: H=%.3f p=%.3f（非参数替代）' % (H, p_kw))
print('✅ 齐性检验与稳健替代完成')

Levene 方差齐性: W=16.801 p=0.000 → 方差不等（需 Welch/非参数）
Kruskal-Wallis: H=20.354 p=0.000（非参数替代）
✅ 齐性检验与稳健替代完成


## 🧮 手算示例：单因素 ANOVA 平方和分解

> **数据**：A=[2,3,4]，B=[4,5,6]，C=[7,8,9]（每组 n=3，共 9 个）
> 组均值：3、5、8；总均值 = 48/9 ≈ 5.333

**第 1 步 · 组间平方和 SSB**：
$$SS_B = 3(3{-}5.333)^2 + 3(5{-}5.333)^2 + 3(8{-}5.333)^2 = 16.33 + 0.33 + 21.33 = 38.0$$

**第 2 步 · 组内平方和 SSW**：
A 组内部：(2-3)²+(3-3)²+(4-3)²=2；B、C 同 → $SS_W = 6$

**第 3 步 · F 统计量**（k=3, n=9, df_B=2, df_W=6）：
$$F = \frac{SS_B/2}{SS_W/6} = \frac{19}{1} = 19,\quad p \ll 0.05$$

**第 4 步 · 效应量**：$\eta^2 = \frac{38}{38+6} = \frac{38}{44} \approx 0.864$（大效应）

In [6]:
import numpy as np
from scipy import stats as st
groups = [np.array([2,3,4]), np.array([4,5,6]), np.array([7,8,9])]
all_d = np.concatenate(groups)
grand = all_d.mean()
ssb = sum(len(g)*(g.mean()-grand)**2 for g in groups)
ssw = sum(((g-g.mean())**2).sum() for g in groups)
k, n = len(groups), len(all_d)
f = (ssb/(k-1)) / (ssw/(n-k))
p = 1 - st.f.cdf(f, k-1, n-k)
eta2 = ssb/(ssb+ssw)
print('SSB(手算=38) =', ssb, '  SSW(手算=6) =', ssw)
print('F(手算=19) =', f, '  p ≈ %.3e' % p)
print('η²(手算≈0.864) =', round(eta2, 3))
f_s, p_s = st.f_oneway(*groups)
assert abs(f - f_s) < 1e-9
assert abs(ssb - 38) < 1e-9 and abs(ssw - 6) < 1e-9
print('✅ 手算与 scipy f_oneway 一致')

SSB(手算=38) = 38.0   SSW(手算=6) = 6.0
F(手算=19) = 19.0   p ≈ 2.536e-03
η²(手算≈0.864) = 0.864
✅ 手算与 scipy f_oneway 一致


## ✅ 自测清单

- [ ] 能手写单因素 ANOVA 并解释 F=MSB/MSW
- [ ] 能算效应量 η² 并解释"为什么不能只看 p"
- [ ] 能用 statsmodels 做双因素 ANOVA 并解释交互作用
- [ ] 能做 Tukey HSD 多重比较
- [ ] 能说出 ANOVA 三大前提与违背时的替代方案